In [1]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

from model_training import load_test_data, train_model, summarize, LABELS
from sentiment_rules import apply_sentiment_rules

In [ ]:
model, vectorizer, _, metrics = train_model()

test = load_test_data()
test['pred_model'] = metrics['predictions']

ruled = [apply_sentiment_rules(t, p) for t, p in zip(test['text'], test['pred_model'])]
test['pred_rules'] = [r[0] for r in ruled]
test['rule_reason'] = ["; ".join(r[1]) for r in ruled]

print(f"Test set: {len(test):,} rows")

Test set: 1,000 rows


In [3]:
comparison = pd.DataFrame({
    "Model only": summarize(test['sentiment'], test['pred_model']),
    "Model + rules": summarize(test['sentiment'], test['pred_rules']),
}).T
comparison.loc["Difference"] = comparison.loc["Model + rules"] - comparison.loc["Model only"]
print(comparison.round(4))

               accuracy  precision_macro  recall_macro  f1_macro
Model only        0.611           0.5973        0.6128    0.6036
Model + rules     0.607           0.5927        0.6052    0.5981
Difference       -0.004          -0.0045       -0.0076   -0.0055


In [4]:
fired = test[test['rule_reason'] != ""]
changed = fired[fired['pred_model'] != fired['pred_rules']]
fixed = (changed['pred_rules'] == changed['sentiment']).sum()
broke = (changed['pred_model'] == changed['sentiment']).sum()

print(f"Rule fired                   : {len(fired):,}")
print(f"Prediction changed by a rule : {len(changed):,}")
print(f"  Wrong -> right (helped)    : {fixed:,}")
print(f"  Right -> wrong (hurt)      : {broke:,}")
print(f"  Wrong -> still wrong       : {len(changed) - fixed - broke:,}")

Rule fired                   : 13
Prediction changed by a rule : 9
  Wrong -> right (helped)    : 2
  Right -> wrong (hurt)      : 6
  Wrong -> still wrong       : 1


In [ ]:
rule_type = changed['rule_reason'].str.extract(
    r"^(Detected neutral phrase|Positive adjective|Detected negation before \w+)"
)[0].rename("rule")
impact = pd.DataFrame({
    "changed": rule_type.value_counts(),
    "helped": rule_type[changed['pred_rules'] == changed['sentiment']].value_counts(),
    "hurt": rule_type[changed['pred_model'] == changed['sentiment']].value_counts(),
}).fillna(0).astype(int)
impact["net"] = impact["helped"] - impact["hurt"]
print(impact.sort_values("net"))

                                   changed  helped  hurt  net
rule                                                         
Detected negation before positive        3       0     2   -2
Detected neutral phrase                  6       2     4   -2


In [6]:
print("Per-class report (model + rules):")
print(pd.DataFrame(classification_report(
    test['sentiment'], test['pred_rules'], labels=LABELS, output_dict=True, zero_division=0
)).transpose().drop(index='accuracy', errors='ignore').round(3))

print("\nConfusion matrix (model + rules):")
print(pd.DataFrame(
    confusion_matrix(test['sentiment'], test['pred_rules'], labels=LABELS),
    index=[f"actual_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS],
))

Per-class report (model + rules):
              precision  recall  f1-score  support
negative          0.605   0.620     0.613    324.0
neutral           0.647   0.605     0.625    483.0
positive          0.525   0.591     0.556    193.0
macro avg         0.593   0.605     0.598   1000.0
weighted avg      0.610   0.607     0.608   1000.0

Confusion matrix (model + rules):
                 pred_negative  pred_neutral  pred_positive
actual_negative            201            99             24
actual_neutral             112           292             79
actual_positive             19            60            114


In [ ]:
broken = changed[changed['pred_model'] == changed['sentiment']]
for _, r in broken.head(15).iterrows():
    print(f"[{r.sentiment}: model {r.pred_model} -> rule {r.pred_rules}] {r.rule_reason}\n   {str(r.text)[:150]}")

[positive: model positive -> rule negative] Detected negation before positive word
   When you want to try vegetarianism but you can't find good recipes and then you stumble upon an awesome-looking vegan site! 😍
[negative: model negative -> rule neutral] Detected neutral phrase: 'fair'
   HA! #Giuliani is a UGE crook! #CrookedRudy never ever trust him, he was in w #Comey #Romey be a fair not a #bigot…
[positive: model positive -> rule neutral] Detected neutral phrase: 'ok'
   I liked a @user video from @user Cristiano Ronaldo .. Do you think he is arrogant? Ok, watch this
[positive: model positive -> rule negative] Detected negation before positive word
   We still love you Tony Romo! You have not been forgotten! Great game boys! #tonyromo #CowboysNation #DallasCowboys #DALvsWAS
[positive: model positive -> rule neutral] Detected neutral phrase: 'okay'
   OKAY if you haven't seen David Blaine's specials on Netflix watch them IMMEDIATELY
[negative: model negative -> rule neutral] Detect